## LSTM Multi-Horizon Training + Visualization
Paste these cells into your existing notebook after your imports, helpers, and class definitions.

In [ ]:
# ============================================================
# TRAINING CELL — Replace your current training cell with this
# Same logic, but stores everything in `results` dict for plots
# ============================================================
import time

HORIZONS = [1, 5, 20]
results = {}  # stores everything for plotting

for h in HORIZONS:
    print(f"\n{'='*60}")
    print(f"  HORIZON = {h} day(s)")
    print(f"{'='*60}")
    t0 = time.time()

    (Xtr, ytr, Xva, yva, Xte, yte,
     dtr, dva, dte, feat_cols) = build_sequences(
        MACRO_FILE, PRICE_FILE, FEATURES,
        lookback=30, horizon=h,
        train_end='2014-12-31', val_end='2024-12-31',
    )

    print(f"\nSequence shapes (h={h}):")
    print(f"  Train: X={Xtr.shape}  y={ytr.shape}  ({dtr.min().date()} → {dtr.max().date()})")
    print(f"  Val  : X={Xva.shape}  y={yva.shape}  ({dva.min().date()} → {dva.max().date()})")
    print(f"  Test : X={Xte.shape}  y={yte.shape}  ({dte.min().date()} → {dte.max().date()})")
    print(f"  y_train — mean: {ytr.mean():.6f}  std: {ytr.std():.6f}")
    print(f"  y_val   — mean: {yva.mean():.6f}  std: {yva.std():.6f}")
    print(f"  y_test  — mean: {yte.mean():.6f}  std: {yte.std():.6f}")

    model = LSTMForecastModel(
        horizon=h, lookback=30, n_splits=5, epochs=200, patience=20,
        batch_size=128,
        hidden_sizes=[32, 64, 128],
        num_layers_list=[1, 2],
        dropouts=[0.1, 0.3],
        learning_rates=[1e-3, 5e-4],
    )
    model.feature_names_ = feat_cols
    model.fit(Xtr, ytr)

    pred_val  = model.predict(Xva)
    pred_test = model.predict(Xte)
    val_metrics  = model.evaluate(Xva, yva)
    test_metrics = model.evaluate(Xte, yte)
    elapsed = time.time() - t0

    results[h] = {
        'Xtr': Xtr, 'ytr': ytr,
        'Xva': Xva, 'yva': yva,
        'Xte': Xte, 'yte': yte,
        'dtr': dtr, 'dva': dva, 'dte': dte,
        'pred_val': pred_val, 'pred_test': pred_test,
        'val_metrics': val_metrics, 'test_metrics': test_metrics,
        'model': model,
        'elapsed': elapsed,
    }

    print(f"Validation (h={h}): {val_metrics}")
    print(f"Test       (h={h}): {test_metrics}")
    print(f"Time: {elapsed/60:.1f} min")
    model.save(f'lstm_wti_h{h}.pkl')

print(f"\n{'='*60}")
print("All horizons complete. Results stored in `results` dict.")
print(f"{'='*60}")

In [ ]:
# ============================================================
# PLOT SETUP — run once before all plot cells
# ============================================================
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.grid': True, 'grid.alpha': 0.3, 'grid.linestyle': '--',
    'font.family': 'serif', 'font.size': 11,
    'axes.titlesize': 14, 'axes.labelsize': 12,
    'legend.fontsize': 10, 'figure.dpi': 150,
})

C = {
    'actual': '#1a1a2e', 'pred': '#e63946', 'fill': '#a8dadc',
    'pos': '#2a9d8f', 'neg': '#e76f51',
    'h1': '#264653', 'h5': '#2a9d8f', 'h20': '#e9c46a',
    'bar1': '#264653', 'bar2': '#e76f51',
}
print('Plot setup done.')

In [ ]:
# ============================================================
# PLOT 1: Predicted vs Actual — Time Series
# ============================================================

for h in HORIZONS:
    r = results[h]
    fig, axes = plt.subplots(2, 1, figsize=(15, 9))

    for ax, dates, yt, yp, label in [
        (axes[0], r['dva'], r['yva'], r['pred_val'], 'Validation'),
        (axes[1], r['dte'], r['yte'], r['pred_test'], 'Test'),
    ]:
        ax.plot(dates, yt, color=C['actual'], lw=0.8, alpha=0.85, label='Actual')
        ax.plot(dates, yp, color=C['pred'], lw=0.8, alpha=0.75, label='Predicted')
        ax.fill_between(dates, yt, yp, alpha=0.10, color=C['fill'])
        ax.axhline(0, color='gray', lw=0.5)
        ax.set_title(f'h={h}d — Predicted vs Actual ({label})', fontweight='bold')
        ax.set_ylabel(f'{h}-Day Log Return')
        ax.legend(loc='upper right', framealpha=0.9)

    axes[1].set_xlabel('Date')
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# PLOT 2: Training & Validation Loss Curves
# ============================================================

def retrain_with_loss_tracking(X_train, y_train, best_params,
                               epochs=200, patience=20, batch_size=128):
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    n = len(X_train)
    split = int(n * 0.9)
    Xtr_f, Xva_f = X_train[:split], X_train[split:]
    ytr_f, yva_f = y_train[:split], y_train[split:]

    _, T, F = Xtr_f.shape
    sc = StandardScaler()
    sc.fit(Xtr_f.reshape(-1, F))
    Xtr_s = sc.transform(Xtr_f.reshape(-1, F)).reshape(len(Xtr_f), T, F).astype(np.float32)
    Xva_s = sc.transform(Xva_f.reshape(-1, F)).reshape(len(Xva_f), T, F).astype(np.float32)

    net = LSTMNet(F, best_params['hidden_size'],
                  best_params['num_layers'], best_params['dropout']).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=best_params['lr'], weight_decay=1e-5)
    crit = nn.MSELoss()

    tr_dl = DataLoader(TensorDataset(torch.tensor(Xtr_s), torch.tensor(ytr_f)),
                       batch_size=batch_size, shuffle=False)
    va_dl = DataLoader(TensorDataset(torch.tensor(Xva_s), torch.tensor(yva_f)),
                       batch_size=batch_size*2, shuffle=False)

    train_losses, val_losses = [], []
    best_val, wait = np.inf, 0

    for ep in range(1, epochs + 1):
        net.train()
        ep_loss = 0.0
        for xb, yb in tr_dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            loss = crit(net(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item() * len(xb)
        train_losses.append(np.sqrt(ep_loss / len(Xtr_s)))

        net.eval()
        vp = []
        with torch.no_grad():
            for xb, _ in va_dl:
                vp.append(net(xb.to(device)).cpu().numpy())
        vl = float(np.sqrt(np.mean((yva_f - np.concatenate(vp))**2)))
        val_losses.append(vl)

        if vl < best_val:
            best_val, wait = vl, 0
        else:
            wait += 1
            if wait >= patience: break

    return train_losses, val_losses


fig, axes = plt.subplots(1, len(HORIZONS), figsize=(6*len(HORIZONS), 5))
if len(HORIZONS) == 1: axes = [axes]

for ax, h in zip(axes, HORIZONS):
    r = results[h]
    bp = r['model'].best_params_
    print(f'Retraining h={h}d for loss curves...')
    tl, vl = retrain_with_loss_tracking(r['Xtr'], r['ytr'], bp)

    eps = range(1, len(tl) + 1)
    ax.plot(eps, tl, color=C['h1'], lw=1.5, label='Train RMSE')
    ax.plot(eps, vl, color=C['neg'], lw=1.5, label='Val RMSE')

    best_ep = np.argmin(vl) + 1
    best_v = min(vl)
    ax.axvline(best_ep, color='gray', ls=':', alpha=0.7)
    ax.annotate(f'Best: ep {best_ep}\n{best_v:.5f}',
                xy=(best_ep, best_v), fontsize=9,
                xytext=(best_ep + len(tl)*0.08, best_v*1.03),
                arrowprops=dict(arrowstyle='->', color='gray'),
                bbox=dict(boxstyle='round,pad=0.3', fc='lightyellow', alpha=0.8))

    ax.set_title(f'h={h}d — Loss Curve', fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('RMSE')
    ax.legend(framealpha=0.9)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT 3: Scatter — Predicted vs Actual
# ============================================================

for h in HORIZONS:
    r = results[h]
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    for ax, yt, yp, label in [
        (axes[0], r['yva'], r['pred_val'], 'Validation'),
        (axes[1], r['yte'], r['pred_test'], 'Test'),
    ]:
        ax.scatter(yt, yp, alpha=0.3, s=12, color=C['h1'], edgecolors='none')

        lims = [min(yt.min(), yp.min()), max(yt.max(), yp.max())]
        m = (lims[1] - lims[0]) * 0.05
        lims = [lims[0]-m, lims[1]+m]
        ax.plot(lims, lims, '--', color='gray', lw=1, alpha=0.7, label='Perfect')

        z = np.polyfit(yt, yp, 1)
        xl = np.linspace(lims[0], lims[1], 100)
        ax.plot(xl, np.poly1d(z)(xl), color=C['pred'], lw=1.5,
                label=f'Fit: y={z[0]:.3f}x + {z[1]:.5f}')

        corr = np.corrcoef(yt, yp)[0, 1]
        ax.text(0.05, 0.95, f'Corr: {corr:.4f}', transform=ax.transAxes,
                fontsize=10, va='top',
                bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.8))

        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_aspect('equal')
        ax.set_title(f'h={h}d — Scatter ({label})', fontweight='bold')
        ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
        ax.legend(loc='lower right', fontsize=9, framealpha=0.9)

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# PLOT 4: Residual Analysis — Histogram + Q-Q
# ============================================================

for h in HORIZONS:
    r = results[h]
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    for row, (yt, yp, label) in enumerate([
        (r['yva'], r['pred_val'], 'Validation'),
        (r['yte'], r['pred_test'], 'Test'),
    ]):
        res = yt - yp

        # Histogram
        ax = axes[row, 0]
        ax.hist(res, bins=50, color=C['h5'], alpha=0.7, edgecolor='white', lw=0.3)
        ax.axvline(0, color=C['neg'], lw=1.5, ls='--')
        ax.axvline(res.mean(), color=C['h1'], lw=1.5, label=f'Mean: {res.mean():.5f}')
        ax.set_title(f'h={h}d — Residuals ({label})', fontweight='bold')
        ax.set_xlabel('Residual (Actual − Predicted)')
        ax.set_ylabel('Count')
        ax.legend(framealpha=0.9)

        # Q-Q
        ax2 = axes[row, 1]
        sr = np.sort(res)
        n = len(sr)
        theo = stats.norm.ppf(np.linspace(0.001, 0.999, n))
        ax2.scatter(theo, sr, alpha=0.4, s=8, color=C['h1'], edgecolors='none')
        q25, q75 = np.percentile(sr, [25, 75])
        t25, t75 = stats.norm.ppf(0.25), stats.norm.ppf(0.75)
        sl = (q75-q25)/(t75-t25)
        ic = q25 - sl*t25
        rx = np.array([theo.min(), theo.max()])
        ax2.plot(rx, sl*rx+ic, '--', color=C['neg'], lw=1.5)
        ax2.set_title(f'h={h}d — Q-Q Plot ({label})', fontweight='bold')
        ax2.set_xlabel('Theoretical Quantiles')
        ax2.set_ylabel('Sample Quantiles')

    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# PLOT 5: Cumulative Returns — Buy & Hold vs LSTM Strategy
# ============================================================

for h in HORIZONS:
    r = results[h]
    fig, axes = plt.subplots(2, 1, figsize=(15, 9))

    for ax, dates, yt, yp, label in [
        (axes[0], r['dva'], r['yva'], r['pred_val'], 'Validation'),
        (axes[1], r['dte'], r['yte'], r['pred_test'], 'Test'),
    ]:
        cum_bh   = np.cumsum(yt)
        cum_long = np.cumsum(np.where(yp > 0, yt, 0))
        cum_ls   = np.cumsum(np.where(yp > 0, yt, -yt))

        ax.plot(dates, cum_bh, color=C['actual'], lw=1.5, label='Buy & Hold')
        ax.plot(dates, cum_long, color=C['pos'], lw=1.5, label='LSTM Long-Only')
        ax.plot(dates, cum_ls, color=C['pred'], lw=1.5, alpha=0.7, label='LSTM Long/Short')
        ax.axhline(0, color='gray', lw=0.5)
        ax.fill_between(dates, cum_bh, alpha=0.05, color=C['actual'])
        ax.set_title(f'h={h}d — Cumulative Returns ({label})', fontweight='bold')
        ax.set_ylabel('Cumulative Log Return')
        ax.legend(loc='best', framealpha=0.9)

    axes[1].set_xlabel('Date')
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# PLOT 6: Rolling Directional Accuracy
# ============================================================

for h in HORIZONS:
    r = results[h]
    fig, axes = plt.subplots(2, 1, figsize=(15, 9))

    for ax, dates, yt, yp, label, win in [
        (axes[0], r['dva'], r['yva'], r['pred_val'], 'Validation', 60),
        (axes[1], r['dte'], r['yte'], r['pred_test'], 'Test',
         max(10, len(r['yte'])//5)),
    ]:
        correct = (np.sign(yt) == np.sign(yp)).astype(float)
        rolling = pd.Series(correct, index=dates).rolling(win, min_periods=win//2).mean()

        ax.plot(dates, rolling, color=C['h1'], lw=1.2)
        ax.axhline(0.5, color=C['neg'], lw=1.5, ls='--', label='50% (Random)')
        ax.fill_between(dates, rolling, 0.5, where=rolling > 0.5,
                        alpha=0.2, color=C['pos'], label='Above 50%')
        ax.fill_between(dates, rolling, 0.5, where=rolling <= 0.5,
                        alpha=0.2, color=C['neg'], label='Below 50%')
        ax.set_ylim(0.2, 0.8)
        ax.set_title(f'h={h}d — Rolling Dir. Accuracy, {win}-day ({label})',
                     fontweight='bold')
        ax.set_ylabel('Directional Accuracy')
        ax.legend(loc='upper right', framealpha=0.9)

    axes[1].set_xlabel('Date')
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# PLOT 7: CV Heatmaps — All Horizons Side by Side
# ============================================================

fig, axes = plt.subplots(1, len(HORIZONS), figsize=(7*len(HORIZONS), 6))
if len(HORIZONS) == 1: axes = [axes]

for ax, h in zip(axes, HORIZONS):
    cv = results[h]['model'].cv_results_
    df = pd.DataFrame(cv)

    df['row'] = 'hs=' + df['hidden_size'].astype(str) + ' nl=' + df['num_layers'].astype(str)
    df['col'] = 'do=' + df['dropout'].astype(str) + ' lr=' + df['lr'].apply(lambda x: f'{x:.0e}')
    pivot = df.pivot(index='row', columns='col', values='cv_rmse')

    im = ax.imshow(pivot.values, cmap='YlOrRd', aspect='auto')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, rotation=45, ha='right', fontsize=8)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=9)

    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            color = 'white' if val > pivot.values.mean() else 'black'
            ax.text(j, i, f'{val:.4f}', ha='center', va='center',
                    fontsize=7, color=color, fontweight='bold')

    bi = np.unravel_index(pivot.values.argmin(), pivot.values.shape)
    ax.add_patch(plt.Rectangle((bi[1]-0.5, bi[0]-0.5), 1, 1,
                 fill=False, edgecolor=C['pos'], lw=3))
    plt.colorbar(im, ax=ax, label='CV RMSE', shrink=0.8)
    ax.set_title(f'h={h}d — CV Heatmap', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PLOT 8: Multi-Horizon Comparison Bar Chart
# ============================================================

met_names = ['rmse', 'mae', 'r2', 'directional_acc']
titles = ['RMSE ↓', 'MAE ↓', 'R² ↑', 'Directional Accuracy ↑']

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
x = np.arange(len(HORIZONS))
w = 0.35

for i, (mn, title) in enumerate(zip(met_names, titles)):
    ax = axes[i]
    vv = [results[h]['val_metrics'][mn] for h in HORIZONS]
    tv = [results[h]['test_metrics'][mn] for h in HORIZONS]

    b1 = ax.bar(x - w/2, vv, w, label='Validation', color=C['bar1'], alpha=0.85)
    b2 = ax.bar(x + w/2, tv, w, label='Test', color=C['bar2'], alpha=0.85)

    ax.set_title(title, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels([f'h={h}d' for h in HORIZONS])

    if mn == 'directional_acc':
        ax.axhline(0.5, color='gray', ls='--', lw=1, alpha=0.7)
    elif mn == 'r2':
        ax.axhline(0, color='gray', ls='--', lw=1, alpha=0.7)

    for bar in list(b1) + list(b2):
        ht = bar.get_height()
        ax.annotate(f'{ht:.3f}', xy=(bar.get_x()+bar.get_width()/2, ht),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=8)
    if i == 0: ax.legend(framealpha=0.9)

plt.suptitle('LSTM Multi-Horizon Comparison', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Summary Metrics Table
# ============================================================

rows = []
for h in HORIZONS:
    for sn, mk in [('Val', 'val_metrics'), ('Test', 'test_metrics')]:
        m = results[h][mk]
        rows.append({
            'Horizon': f'{h}d', 'Split': sn,
            'RMSE': f"{m['rmse']:.6f}", 'MAE': f"{m['mae']:.6f}",
            'R²': f"{m['r2']:.6f}", 'Dir. Acc': f"{m['directional_acc']:.4f}",
        })

summary = pd.DataFrame(rows)
print('\n' + '='*65)
print('  LSTM Multi-Horizon Results Summary')
print('='*65)
print(summary.to_string(index=False))
print('='*65)

print('\nBest Hyperparameters:')
for h in HORIZONS:
    bp = results[h]['model'].best_params_
    print(f"  h={h:>2}d: hs={bp['hidden_size']}, nl={bp['num_layers']}, "
          f"do={bp['dropout']}, lr={bp['lr']}, cv_rmse={bp['cv_rmse']:.6f}")